# 08 — Real Medical Dataset Benchmark

This notebook is the bridge from the deterministic development benchmark to a real medical-image evaluation. It does not invent results: every number is generated from the supplied manifest and saved protocol.

## Research objective
Compare identical zero-watermarking algorithms on the same medical images, attacks, hash length and split. Report robustness, discriminability and hash-quality metrics separately.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from zero_watermarking.datasets import load_manifest, validate_manifest, group_split
from zero_watermarking.real_benchmark import run_manifest_benchmark


## 1. Choose the dataset

Create `data/manifests/<dataset>.csv` using the schema documented in `data/README.md`. For a journal study, use patient/study identifiers in `group_id` and never split related slices across partitions.

Recent medical zero-watermarking work has evaluated chest X-ray and other medical modalities, including multi-scale and deep-feature approaches. The benchmark should therefore include strong classical and learning-based baselines before claiming an improvement.


In [ ]:
MANIFEST = 'data/manifests/your_dataset.csv'   # change this
ROOT_DATA = None                              # e.g. '/datasets/chestxray14'

records = load_manifest(MANIFEST)
manifest = validate_manifest(records, ROOT_DATA)
display(manifest.head())
print('images:', len(manifest))
print('groups:', manifest['group_id'].nunique())
print('modalities:', manifest['modality'].value_counts(dropna=False).to_dict())


In [ ]:
train, val, test = group_split(manifest, test_size=0.20, val_size=0.10, seed=42)
print('train:', len(train), 'images /', train.group_id.nunique(), 'groups')
print('val:  ', len(val), 'images /', val.group_id.nunique(), 'groups')
print('test: ', len(test), 'images /', test.group_id.nunique(), 'groups')
assert set(train.group_id).isdisjoint(val.group_id)
assert set(train.group_id).isdisjoint(test.group_id)
assert set(val.group_id).isdisjoint(test.group_id)


## 2. Development run
Start with a small fixed subset so the pipeline can be debugged. This is not the final paper result.


In [ ]:
dev = run_manifest_benchmark(
    manifest_path=MANIFEST,
    root=ROOT_DATA,
    image_size=224,
    hash_length=256,
    seed=42,
    max_images=100,
    out_dir='experiments/results/real/dev',
)
display(dev.sort_values('mean_nc', ascending=False))


## 3. Robustness comparison
The exact column names depend on the evaluator; inspect the saved summary before generating paper figures. Never hard-code an improvement percentage.


In [ ]:
summary_path = Path('experiments/results/real/dev/benchmark_summary.csv')
results = pd.read_csv(summary_path)
display(results)

ax = results.plot.bar(x='method', y='mean_nc', legend=False, figsize=(10,5))
ax.set_ylabel('Mean NC (higher is better)')
ax.set_xlabel('Method')
ax.set_title('Development robustness benchmark')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 4. What counts as a meaningful improvement?

A publishable claim should survive all of these checks:

- clean/attacked **NC increases** and **BER decreases**;
- maximum **intra-image Hamming distance decreases**;
- minimum **inter-image Hamming distance increases**;
- **AUC increases** and **EER decreases**;
- the **collision gap** moves upward rather than improving robustness at the cost of discrimination;
- bit balance and correlation remain acceptable;
- gains persist over multiple seeds and the same attack grid;
- confidence intervals and paired significance tests are reported.


## 5. Final-paper checklist

Run the full protocol only after the development run passes. Save the manifest checksum, dataset version, preprocessing, split seed, attack grid, hash length, software versions and random seeds alongside the results. Keep literature-reported numbers separate from reproduced numbers.
